In [2]:
#Hybrid RETRIEVER-cOMBINING  dense and sparse retriever

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever

In [41]:
#Step-1:Taking Sample documents
docs=[
    Document(page_content="Langchain Helps in building LLM Applications"),
    Document(page_content="Pinecone is a vector database for semantic search"),
    Document(page_content="The Eiffel Tower is located in Paris"),
    Document(page_content="The Eiffel Tower is located in Paris"),
    Document(page_content="Langchain can be used to develop agentic AI Applications")
]

In [42]:
#Dense Rtreiver(FAISS+HuggingFace)
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)
dense_vectorstore=FAISS.from_documents(docs,embeddings)
dense_retriever=dense_vectorstore.as_retriever()

In [43]:
###Sparse Retrivever
sparse_retriever=BM25Retriever.from_documents(docs)

#Combine with ensemble retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weights=[0.7,0.3]
)

In [7]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002525A689B50>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x00000252008C75C0>)], weights=[0.7, 0.3])

In [8]:
query="How can I Build an Application using LLMS?"
result=hybrid_retriever.invoke(query)

In [9]:
result

[Document(id='63a75585-c50d-4887-85d5-cd1e1957b43a', metadata={}, page_content='The Eiffel Tower is located in Paris'),
 Document(id='923f5d41-3fe8-4b73-8e10-51a91f46bc80', metadata={}, page_content='Langchain can be used to develop agentic AI Applications'),
 Document(id='1e93a137-bd12-4516-8c42-d95fa8897bb4', metadata={}, page_content='Pinecone is a vector database for semantic search'),
 Document(id='2e538086-9004-40a7-a061-386587b70f09', metadata={}, page_content='Langchain Helps in building LLM Applications')]

In [11]:
#Step-6-Print Result
for i,doc in enumerate(result):
    print(f"Document {i+1}\n{doc.page_content}")

Document 1
The Eiffel Tower is located in Paris
Document 2
Langchain can be used to develop agentic AI Applications
Document 3
Pinecone is a vector database for semantic search
Document 4
Langchain Helps in building LLM Applications


RAG PIPELINE WITH HYBRID RETRIEVER

In [37]:
from langchain_groq.chat_models import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain.chat_models import init_chat_model

In [38]:
#Step5-Prompt Template
prompt=PromptTemplate.from_template("""
Answer the question based on the context Below
Context:{context}
Question:{input}
""")

In [39]:
#Step6-llm-modle
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.4)

In [40]:
###Create Studd Document Chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

In [44]:
#Create an full rag chain
from langchain_classic.chains import create_retrieval_chain
rag_chain=create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=document_chain
)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002527E40F500>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000002522BC56AE0>)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context Below\nContext:{context}\nQuestion:{input}\n')
            | ChatGroq(profile={'max_input_to

In [46]:
query={"input":"How can I Build an app using LLMs?"}
response=rag_chain.invoke(query)
print(f"Answer: {response['answer']}")

Answer: To build an app using LLMs (Large Language Models), you'll need to integrate them with a suitable architecture. Based on the context, I'll outline a high-level approach using Langchain, which can help build LLM applications.

Here's a step-by-step guide:

1. **Choose a Langchain framework**: Langchain provides a suite of tools for building LLM applications. You can use their pre-built frameworks or create your own custom architecture.
2. **Select a suitable LLM model**: Langchain supports various LLM models, such as BERT, RoBERTa, and XLNet. Choose a model that fits your app's requirements.
3. **Integrate with a vector database**: To enable semantic search, consider using a vector database like Pinecone. This will allow your app to search and retrieve relevant information using vector embeddings.
4. **Design your app's architecture**: Determine the flow of your app's interactions with the LLM model. This might involve text input, processing, and output generation.
5. **Implemen